# Morgan & Druckmüller (2014) — Multi-Scale Gaussian Normalization (MGN) / 구현

**Paper**: Morgan, H. & Druckmüller, M. (2014). *Multi-Scale Gaussian Normalization for Solar Image Processing.* Solar Physics, **289**(8), 2945–2955.

This notebook implements the paper's MGN algorithm exactly as specified in Section 3 (Eqs. 1–5 + 10-step pseudocode):

1. **Local Gaussian normalisation** at one scale (Eqs. 1–2).
2. **arctan compression** (Eq. 3).
3. **Multi-scale loop** over $w \in \{1.25, 2.5, 5, 10, 20, 40\}$ pixels.
4. **Global γ correction** (Eq. 4) and **final blend** (Eq. 5).
5. **Demonstration** on a synthetic "AIA-like" image (or `skimage.data.sun()` / `skimage.data.camera()` if no solar data is at hand).
6. **Scale-weight study** reproducing the qualitative shape of paper Fig. 4.
7. **Summary table** matching the paper's recipe to modern equivalents.

이 노트북은 식 1–5 와 의사코드 10 단계를 그대로 구현해 합성 "AIA-like" 영상 (또는 skimage 의 sun / camera 영상) 으로 검증한다.

In [ ]:
from __future__ import annotations

import time
from typing import Sequence

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

try:
    from skimage import data, exposure
    HAVE_SKIMAGE = True
except ImportError:
    HAVE_SKIMAGE = False

rng = np.random.default_rng(2026)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## Part 1: Build a synthetic AIA-like image / 합성 AIA-스타일 영상 만들기

We construct an image with the characteristic dynamic range of an EUV solar image: a few very bright "active region" patches dominate (intensity $\sim 10^4$), surrounded by a faint quiet-Sun field ($\sim 10^2$), and an off-limb region ($\sim 30$) that contains delicate filamentary structure. This reproduces the four-population histogram of the paper's Fig. 2.

In [ ]:
def make_synthetic_aia(h: int = 256, w: int = 256, seed: int = 1) -> np.ndarray:
    """Generate a synthetic AIA-like image with extreme dynamic range.

    Args:
        h: Height in pixels.
        w: Width in pixels.
        seed: Reproducible seed.

    Returns:
        2-D image with active regions, quiet Sun, off-limb structures, and Poisson noise.
    """
    g = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:h, 0:w]
    cy, cx = h / 2, w / 2
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    R_disk = 0.4 * min(h, w)

    disk_mask = r < R_disk
    quiet = 100.0 * np.exp(-((r / R_disk) ** 2) * 0.5) * disk_mask

    # active regions / 활동 영역
    img = quiet.copy()
    for (ay, ax, br, ba) in [(0.55, 0.45, 14, 8000.0), (0.40, 0.55, 8, 4000.0), (0.50, 0.65, 6, 5000.0)]:
        ay_p, ax_p = ay * h, ax * w
        rr = np.sqrt((yy - ay_p) ** 2 + (xx - ax_p) ** 2)
        img += ba * np.exp(-(rr ** 2) / (2 * br ** 2))

    # off-limb fan-like loops / off-limb 부채꼴 구조
    angle = np.arctan2(yy - cy, xx - cx)
    fan = 25.0 * np.exp(-((r - R_disk * 1.10) ** 2) / (2 * 4.0 ** 2)) * (np.cos(8 * angle) ** 2)
    fan += 15.0 * np.exp(-((r - R_disk * 1.18) ** 2) / (2 * 5.0 ** 2)) * (np.cos(11 * angle) ** 2)
    img = np.where(r > R_disk, np.maximum(img, fan), img)

    # photon noise / 광자 잡음
    img = g.poisson(np.clip(img, 0, None)).astype(np.float64)
    img = img + 1.0  # avoid zeros
    return img


img = make_synthetic_aia(h=256, w=256, seed=1)
print(f'image shape  = {img.shape}')
print(f'image min/max = {img.min():.2f} / {img.max():.2f}')
print(f'image mean    = {img.mean():.2f}, median = {np.median(img):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(img, cmap='inferno', vmin=0, vmax=img.max())
axes[0].set_title('linear scale (raw)')
axes[1].imshow(np.sqrt(np.clip(img, 0, None)), cmap='inferno')
axes[1].set_title('square-root scale (paper Fig. 1 right)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## Part 2: Single-scale Gaussian normalisation / 단일 스케일 정규화 (Eqs. 1–2)

$$
C = \frac{B - B \otimes k_w}{\sigma_w}, \qquad \sigma_w = \sqrt{\big[(B - B \otimes k_w)^2\big] \otimes k_w}.
$$

We use `scipy.ndimage.gaussian_filter`, which is internally separable, for the 2-D Gaussian convolution.

In [ ]:
def local_gauss_normalise(B: np.ndarray, w: float, eps: float = 1e-6) -> np.ndarray:
    """Locally normalise an image by Gaussian-weighted local mean and std.

    Implements Eqs. (1)–(2) of Morgan & Druckmüller (2014).

    Args:
        B: 2-D input image (must be non-negative; replace negatives upstream).
        w: One-sigma Gaussian width in pixels.
        eps: Floor on the local std to avoid division by zero in faint regions.

    Returns:
        Locally normalised image C with approximate mean 0 and std 1.
    """
    local_mean = gaussian_filter(B, sigma=w, mode='reflect')
    delta = B - local_mean
    local_var = gaussian_filter(delta * delta, sigma=w, mode='reflect')
    sigma_w = np.sqrt(np.maximum(local_var, eps))
    return delta / sigma_w


C_w20 = local_gauss_normalise(img, w=20.0)
print(f'C @ w=20: mean = {C_w20.mean():+.3f}, std = {C_w20.std():.3f}')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(np.sqrt(np.clip(img, 0, None)), cmap='inferno'); axes[0].set_title('sqrt(B)'); axes[0].axis('off')
vlim = max(abs(C_w20.min()), abs(C_w20.max()))
axes[1].imshow(C_w20, cmap='RdBu_r', vmin=-vlim, vmax=vlim)
axes[1].set_title('C  (Eq. 1, w = 20 px)')
axes[1].axis('off')
plt.tight_layout(); plt.show()

## Part 3: arctan compression and full MGN / arctan 압축과 전체 MGN (Eqs. 3–5)

The full pipeline is: locally normalise at multiple scales $w_i$, arctan-compress, blend the multi-scale mean with a global γ-corrected image.

In [ ]:
def gamma_correct(B: np.ndarray, gamma: float = 3.2) -> np.ndarray:
    """Apply a global γ correction (Eq. 4).

    Args:
        B: 2-D image, assumed non-negative.
        gamma: Exponent; γ > 1 brightens midtones.

    Returns:
        Image scaled to [0, 1] then raised to 1/γ.
    """
    a0, a1 = float(B.min()), float(B.max())
    if a1 <= a0:
        return np.zeros_like(B)
    scaled = (B - a0) / (a1 - a0)
    return np.power(np.clip(scaled, 0.0, 1.0), 1.0 / gamma)


def mgn(
    B: np.ndarray,
    widths: Sequence[float] = (1.25, 2.5, 5.0, 10.0, 20.0, 40.0),
    k: float = 0.7,
    h: float = 0.7,
    gamma: float = 3.2,
    weights: Sequence[float] | None = None,
    return_components: bool = False,
) -> np.ndarray:
    """Multi-Scale Gaussian Normalization (Morgan & Druckmüller 2014).

    Implements the 10-step pseudocode of Section 3.1 (Eqs. 1–5).

    Args:
        B: 2-D image; spurious negatives should be replaced upstream.
        widths: Gaussian one-sigma widths in pixels (paper default).
        k: arctan compression factor (Eq. 3).
        h: Global-tone weight (Eq. 5).
        gamma: γ for the global tone image (Eq. 4).
        weights: Per-scale weights g_i (default uniform 1.0).
        return_components: If True, also return the per-scale C_i' images.

    Returns:
        Final MGN image I, plus components if requested.
    """
    if weights is None:
        weights = [1.0] * len(widths)
    if len(weights) != len(widths):
        raise ValueError('weights must match the number of widths')
    Bp = np.where(B < 0, 0.0, B).astype(np.float64)
    components: list[np.ndarray] = []
    accum = np.zeros_like(Bp)
    for w, g in zip(widths, weights):
        C_i = local_gauss_normalise(Bp, w=w)
        Cp_i = np.arctan(k * C_i)
        accum = accum + g * Cp_i
        components.append(Cp_i)
    accum = accum / len(widths)
    Cg = gamma_correct(Bp, gamma=gamma)
    # Bring multi-scale term to a comparable [0, 1]-ish range / 다중스케일 항을 0–1 범위로
    # arctan range is roughly [-π/2, π/2]; rescale to [0, 1] for comparable blending
    accum_scaled = (accum + np.pi / 2) / np.pi
    out = h * Cg + (1 - h) * accum_scaled
    if return_components:
        return out, components, Cg
    return out


t0 = time.perf_counter()
I, comps, Cg = mgn(img, return_components=True)
elapsed = time.perf_counter() - t0
print(f'MGN on {img.shape} took {elapsed:.3f} s (paper: ∼1 s for 500×500)')
print(f'Output range: [{I.min():.3f}, {I.max():.3f}]')

## Part 4: Visual comparison and per-scale components / 시각적 비교와 스케일별 성분

The full MGN output should reveal both the bright-region fine structure and the off-limb / quiet-Sun details that the raw and sqrt views hide.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
panels = [
    (np.sqrt(np.clip(img, 0, None)), 'sqrt(B)  — paper Fig. 1 right'),
    (Cg, 'global γ image C_g'),
    (np.mean(comps, axis=0), 'mean of C_i\'  (multi-scale, locally normalised)'),
    (I, 'MGN final  I  (Eq. 5)'),
]
for ax, (im, ttl) in zip(axes, panels):
    ax.imshow(im, cmap='inferno')
    ax.set_title(ttl)
    ax.axis('off')
plt.tight_layout(); plt.show()

# Per-scale view / 스케일별 시각화
widths = [1.25, 2.5, 5.0, 10.0, 20.0, 40.0]
fig, axes = plt.subplots(1, len(widths), figsize=(2.4 * len(widths), 3))
for ax, c, w in zip(axes, comps, widths):
    ax.imshow(c, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
    ax.set_title(f'w = {w} px')
    ax.axis('off')
plt.tight_layout(); plt.show()

## Part 5: Reproducing Fig. 4 — local-σ vs kernel width / Fig. 4 재현

The paper's Fig. 4 shows that for a pure-noise image, $\langle\sigma_w\rangle / \sigma_{\rm global}$ rises from $\sim 0.6$ at $w \approx 0.6$ px to $\sim 1.0$ for $w \gtrsim 5$ px. We reproduce this on a $256\times 256$ Gaussian-noise field. The result motivates choosing per-scale weights $g_i$ that are smaller for fine-scale kernels.

In [ ]:
noise = rng.standard_normal((256, 256))
sigma_global = float(noise.std())
scan_widths = [0.625, 1.2, 2.5, 5.0, 10.0, 20.0, 30.0]
ratios = []
for w in scan_widths:
    local_mean = gaussian_filter(noise, sigma=w, mode='reflect')
    local_var = gaussian_filter((noise - local_mean) ** 2, sigma=w, mode='reflect')
    ratios.append(float(np.sqrt(local_var).mean()) / sigma_global)

plt.figure(figsize=(7, 4))
plt.semilogx(scan_widths, ratios, 'o-')
plt.axhline(1.0, color='k', ls=':')
plt.xlabel('Gaussian one-sigma width w (px)')
plt.ylabel(r'$\langle\sigma_w\rangle / \sigma_\mathrm{global}$')
plt.title('Local σ stabilisation — reproduces paper Fig. 4 trend')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
for w, r in zip(scan_widths, ratios):
    print(f'w = {w:6.3f} px:  ⟨σ_w⟩ / σ_global = {r:.3f}')

## Part 6: Effect of scale weights $g_i$ / 스케일 가중치의 효과

Compare uniform $g_i = 1$ versus per-scale $g_i$ derived from Fig. 4 (smaller weights for fine-scale, noise-dominated bands). The latter should produce a slightly cleaner image in the off-limb region while preserving large-scale structure.

In [ ]:
widths = [1.25, 2.5, 5.0, 10.0, 20.0, 40.0]
# per-scale weights using the reproduced Fig. 4 trend / 가중치 곡선 사용
g_fig4 = [0.62, 0.78, 0.94, 0.99, 1.0, 1.0]
I_uniform = mgn(img, widths=widths, weights=[1.0] * len(widths))
I_weighted = mgn(img, widths=widths, weights=g_fig4)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(I_uniform, cmap='inferno'); axes[0].set_title('MGN, g_i = 1 uniform'); axes[0].axis('off')
axes[1].imshow(I_weighted, cmap='inferno'); axes[1].set_title('MGN, g_i ∝ ⟨σ_w⟩/σ_global'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Part 7: Computational scaling check / 계산 비용 측정

Time MGN on increasing image sizes; the paper claims linear scaling with pixel count for separable Gaussian filtering. Verify the trend on $128 \to 512$ pixel images (we keep this conservative for CPU runtime).

In [ ]:
sizes = [128, 192, 256, 384, 512]
times = []
for s in sizes:
    img_s = make_synthetic_aia(h=s, w=s, seed=1)
    t0 = time.perf_counter()
    _ = mgn(img_s)
    times.append(time.perf_counter() - t0)
    print(f'{s:4d}×{s:<4d} ({s*s:>7,d} px): {times[-1]:6.3f} s')

plt.figure(figsize=(7, 4))
plt.loglog([s * s for s in sizes], times, 'o-', label='measured')
ref_x = np.array([s * s for s in sizes])
ref_y = times[0] * (ref_x / ref_x[0])
plt.loglog(ref_x, ref_y, 'k--', alpha=0.6, label='linear in pixel count')
plt.xlabel('number of pixels'); plt.ylabel('time (s)')
plt.title('MGN scaling — paper claims ~ 40 s for 4096×4096')
plt.grid(alpha=0.3, which='both'); plt.legend(); plt.tight_layout(); plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Local-mean removal / 국지 평균 제거 | $B - B \otimes k_w$ (Eq. 1 numerator) | High-pass filter; difference-of-Gaussians (DoG); CNN U-Net skip connections |
| Local-std normalisation / 국지 표준편차 정규화 | $\sigma_w$ from squared residual (Eq. 2) | Local response normalisation in CNNs; CLAHE bin std |
| Output-range control / 출력 범위 통제 | arctan compression $C' = \arctan(kC)$ | tanh / sigmoid / Lambert W output activations |
| Multi-scale combination / 다중스케일 결합 | weighted mean of $C_i'$ over $w_i$ | Multiscale CNN towers; Inception modules; pyramid pooling |
| Global tone preservation / 전역 톤 유지 | γ-corrected $C_g$ + weight $h$ | Tone-mapping operators in HDR processing |
| Computational efficiency / 계산 효율 | Separable Gaussians, $\mathcal O(N\, n\, w_{\max})$ | Separable / depthwise-separable convolutions in MobileNet, EfficientNet |
| Solar applications / 태양 응용 | AIA, Hi-C, SWAP, LASCO | Default in Helioviewer, JHelioviewer, SunPy, EUI/IRIS pipelines |